### 01_bronzeingestion.ipynb
* BRONZE LAYER - PATH-BASED DELTA LAKE (NON-UC) \
    - `cases` 
    - `courts`
* Approach  : External Storage with Direct Path Access / No UNITY CATALOG
* Source    : ADLS Landing Zone (CSV)
* Target    : ADLS Bronze Container (Delta)
* Mode      : Cases → Append (Incremental) | Courts → Overwrite (Full Load) \


In [0]:
#IMPORTS
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.functions import current_timestamp, input_file_name
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType,
                                DateType, TimestampType, BooleanType)

In [0]:
spark = SparkSession.builder \
            .config('spark.sql.shuffle.partitions', '20') \
            .getOrCreate()

In [0]:
#=========================Storaeg Access for Non UC==================
# Before this make sure secret scope added & kv has the storage secrets

storage_account = "saadlsecourtsindia"
client_id = dbutils.secrets.get(scope="kv-ecourtsproject",key="sp-client-id")
client_secret = dbutils.secrets.get(scope="kv-ecourtsproject",key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope="kv-ecourtsproject",key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net","OAuth")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",client_secret)
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")


In [0]:
#=============================PATHS==================================
path_landing_cases  = "abfss://landing@saadlsecourtsindia.dfs.core.windows.net/cases/"
path_landing_courts = "abfss://landing@saadlsecourtsindia.dfs.core.windows.net/courts/"
path_bronze_cases   = "abfss://bronze@saadlsecourtsindia.dfs.core.windows.net/cases/"
path_bronze_courts  = "abfss://bronze@saadlsecourtsindia.dfs.core.windows.net/courts/"

### Cases

In [0]:
#================================CASES=======================================
# GOAL: LANDING (CSV) --> BRONZE (DELTA)

cases_schema = StructType([
    StructField("case_id",       IntegerType(),   nullable=False),
    StructField("case_type",     StringType(),    nullable=True),
    StructField("case_subtype",  StringType(),    nullable=True),
    StructField("filing_date",   DateType(),      nullable=True),
    StructField("status",        StringType(),    nullable=True),
    StructField("court_id",      IntegerType(),   nullable=True),
    StructField("last_modified", TimestampType(), nullable=True),
    StructField("is_deleted",    BooleanType(),   nullable=True)
])

# Read
cases_df = spark.read \
            .option("header", "true") \
            .schema(cases_schema) \
            .csv(path_landing_cases)

# Add AUDITING columns
cases_df =  cases_df \
            .withColumn("bronze_ingested_at", current_timestamp()) \
            .withColumn("source_file", col("_metadata.file_path"))



In [0]:
# Write to Bronze - Append via (Incremental Load) using last_modified > last_watermark
cases_df.write   \
    .format("delta") \
    .mode("append") \
    .save(path_bronze_cases)

print(f"Cases Table Successfully Written in Bronze: {cases_df.count()} rows")

Cases Table Successfully Written in Bronze: 188 rows


### Courts

In [0]:
#================================COURTS=======================================
# GOAL: LANDING (CSV) --> BRONZE (DELTA)

courts_schema = StructType([
    StructField("court_id",        IntegerType(), nullable=False),
    StructField("court_name",      StringType(),  nullable=True),
    StructField("court_level",     StringType(),  nullable=True),
    StructField("parent_court_id", IntegerType(), nullable=True),
    StructField("bench_name",      StringType(),  nullable=True),
    StructField("state_name",      StringType(),  nullable=True),
    StructField("judge_count",     IntegerType(), nullable=True)
])

# Read
files = dbutils.fs.ls(path_landing_courts)  # reading all the files
latest_courts_file = sorted(files, key=lambda f: f.modificationTime)[-1].path   # fetching latest

courts_df = spark.read \
            .option("header", "true") \
            .schema(courts_schema) \
            .csv(latest_courts_file) 

# Add audit columns
courts_df = courts_df   \
            .withColumn("bronze_ingested_at", current_timestamp()) \
            .withColumn("source_file", col("_metadata.file_path"))


In [0]:
courts_df.write   \
        .format("delta") \
        .mode("overwrite") \
        .save(path_bronze_courts)

print(f"Courts Table Successfully Written in Bronze: {courts_df.count()} rows")

Courts Table Successfully Written in Bronze: 50 rows


In [0]:
from delta.tables import DeltaTable
bronze_courts_path = "abfss://bronze@saadlsecourtsindia.dfs.core.windows.net/courts/"
DeltaTable.forPath(spark, bronze_courts_path).history().show(10, truncate=False)

+-------+-------------------+---------------+----------------------+---------+------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+--------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId         |userName              |operation|operationParameters                                         |job                                                                                                                                                                             |notebook          |clusterId           |readVersion|isolationLevel   |isBlindAppend|operati